# XDM / MFEX: обзор методов безмодельного объяснимого анализа данных

**XDM = Explainable Data Mining**.  
**MFEX = Model-Free Explainability Framework**.

Этот Colab — теоретический и методический. Его задача — объяснить студентам, какие именно безмодельные методы используются в проекте, что каждый метод измеряет, как интерпретировать результаты и какие ограничения надо указать в отчёте.

## Жёсткое правило проекта

В XDM-протоколе мы не используем обучаемую предсказательную модель как источник объяснения.

Запрещено в объяснительной части:

- нейросети и эпохи обучения;
- Random Forest / XGBoost / Logistic Regression как объяснитель;
- SHAP/LIME поверх обученной модели;
- kNN-классификатор как объяснитель;
- любые действия, где качество объяснения зависит от обученной предсказательной модели.

Разрешено:

- статистические зависимости;
- расстояния и локальная геометрия данных;
- условные зависимости и причинно-ориентированные скрининги;
- интерпретируемые правила и подгруппы;
- bootstrap/permutation-проверки;
- ранговый консенсус.

## 1. Чем XDM отличается от EDA и XAI

### EDA
Exploratory Data Analysis отвечает на вопрос:

> Что видно в данных?

Например: распределения, пропуски, корреляции, выбросы.

### XAI
Explainable AI обычно отвечает на вопрос:

> Почему обученная модель дала такое предсказание?

То есть сначала есть модель, затем объяснение модели.

### XDM
Explainable Data Mining отвечает на вопрос:

> Какие признаки являются важными для структуры самих данных без обращения к обученной предсказательной модели?

XDM объясняет не модель, а данные: статистическую зависимость, геометрию, условные связи, правила и устойчивость признаков.

## 2. Общая архитектура MFEX

MFEX агрегирует четыре независимых источника информации о важности признаков:

| Блок | Что измеряет | Тип объяснения | Выход |
|---|---|---|---|
| Statistical | связь признака с целевой переменной | зависимость | вектор важности |
| Geometric | разделимость классов в пространстве признаков | геометрия | вектор важности |
| Causal-screening | устойчивость условной связи с target | причинно-ориентированный скрининг | вектор важности |
| Structural | наличие признака в сильных правилах | правила / подгруппы | вектор важности |

Далее все важности переводятся в ранги. Итоговое объяснение строится как **медианный консенсусный ранг** с отсечением нестабильных признаков по IQR.

## 3. Метод 1: Statistical Importance

### 3.1. Mutual Information

Показывает, насколько знание признака уменьшает неопределённость относительно целевой переменной:

$$
I(X;Y)=\sum_{x,y} p(x,y)\lograc{p(x,y)}{p(x)p(y)}
$$

### 3.2. Relief-style contrast

хороший признак должен быть похожим у объектов одного класса и различаться у объектов разных классов.

Для каждого объекта ищем:

- ближайших соседей своего класса;
- ближайших соседей другого класса.

Признак получает высокий score, если различия с объектами другого класса больше, чем различия с объектами своего класса.


In [ ]:
# Toy example: Mutual Information + Relief-style contrast without model training
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import pairwise_distances

X_toy = pd.DataFrame({
    'x_signal': [0, 0, 1, 1, 5, 6, 6, 7],
    'x_noise':  [4, 1, 3, 2, 4, 1, 3, 2]
})
y_toy = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

mi = mutual_info_classif(X_toy, y_toy, random_state=42)
pd.Series(mi, index=X_toy.columns, name='mutual_information')

## 4. Метод 2: Geometric Importance

В строгой версии проекта мы не используем `SHAP@1NN`, потому что SHAP обычно требует предиктор, а 1-NN в библиотеках часто вызывается через интерфейс обучения.

Вместо этого используем **геометрическую важность без модели**.

### Идея

Признак важен, если по нему объекты разных классов геометрически разделяются сильнее, чем объекты одного класса.

Для каждого признака можно считать:

$$
G_j = rac{D_{between}(X_j)}{D_{within}(X_j)+\epsilon}
$$

где:

- $D_{between}$ — средняя дистанция между объектами разных классов;
- $D_{within}$ — средняя дистанция между объектами одного класса.

### Интерпретация

Высокий score означает, что признак создаёт разделение target в геометрии данных. Это не модель и не классификатор — только анализ расстояний.

In [ ]:
def geometric_importance_univariate(X, y):
    scores = {}
    y_arr = np.asarray(y)
    for col in X.columns:
        values = np.asarray(X[col], dtype=float).reshape(-1, 1)
        d = pairwise_distances(values, values)
        same = y_arr[:, None] == y_arr[None, :]
        diff = ~same
        np.fill_diagonal(same, False)
        within = d[same].mean() if np.any(same) else 0.0
        between = d[diff].mean() if np.any(diff) else 0.0
        scores[col] = between / (within + 1e-9)
    return pd.Series(scores)

geometric_importance_univariate(X_toy, y_toy)

## 5. Метод 3: Causal-screening Importance

Полноценное causal discovery требует сильных предпосылок: линейность, отсутствие скрытых конфаундеров, корректные conditional independence tests. Поэтому в базовом XDM-протоколе мы используем более осторожную формулировку:

> не «мы нашли истинную причинность», а «мы нашли признаки с устойчивой условной зависимостью с target».

### Идея

Признак важен, если его связь с target сохраняется даже после контроля нескольких других признаков.

Для этого используется partial correlation / условная зависимость через корреляционную матрицу:

$$

ho_{XY \cdot Z} = -rac{P_{XY}}{\sqrt{P_{XX}P_{YY}}}
$$

где $P$ — precision matrix, обратная к корреляционной матрице переменных $X, Y, Z$.

### Интерпретация

Если признак теряет связь с target после контроля других признаков, он может быть proxy или следствием confounding. Если связь устойчива, признак получает более высокий causal-screening score.

In [ ]:
def partial_corr_from_frame(frame, x_col, y_col, z_cols=None):
    z_cols = z_cols or []
    cols = [x_col, y_col] + list(z_cols)
    arr = frame[cols].to_numpy(dtype=float)
    corr = np.corrcoef(arr, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0)
    precision = np.linalg.pinv(corr)
    denom = np.sqrt(max(precision[0, 0] * precision[1, 1], 1e-12))
    return float(-precision[0, 1] / denom)

toy_frame = X_toy.copy()
toy_frame['target'] = y_toy
partial_corr_from_frame(toy_frame, 'x_signal', 'target', z_cols=['x_noise'])

## 6. Метод 4: Structural Importance через Subgroup Discovery

Subgroup Discovery ищет правила вида:

> IF условие по признакам THEN target распределён необычно.

Пример:

> IF `radius_mean > 1.2` AND `concavity_mean > 0.7` THEN доля класса 0 выше базовой.

### WRAcc

Weighted Relative Accuracy:

$$
WRAcc = coverage(rule) \cdot |rate(rule)-rate(data)|
$$

Где:

- `coverage` — доля объектов, покрытых правилом;
- `rate(rule)` — частота target=1 внутри подгруппы;
- `rate(data)` — базовая частота target=1.

### Важность признака

Признак важен, если он часто входит в top-правила или входит в правила с высоким WRAcc.

In [ ]:
def simple_single_feature_rules(X, y, n_bins=3):
    y_arr = np.asarray(y).astype(int)
    base_rate = y_arr.mean()
    rules = []
    for col in X.columns:
        bins = pd.qcut(X[col], q=n_bins, duplicates='drop')
        for level in bins.dropna().unique():
            mask = (bins == level).to_numpy()
            if mask.sum() == 0:
                continue
            rate = y_arr[mask].mean()
            coverage = mask.mean()
            wracc = coverage * abs(rate - base_rate)
            rules.append({'feature': col, 'condition': str(level), 'coverage': coverage, 'rate': rate, 'wracc': wracc})
    return pd.DataFrame(rules).sort_values('wracc', ascending=False)

simple_single_feature_rules(X_toy, y_toy)

## 7. Консенсусное ранжирование

Каждый метод даёт свой вектор важности. Чтобы их сравнить, переводим важности в ранги:

- ранг 1 — самый важный признак;
- чем больше ранг, тем ниже важность.

### Медианный консенсус

$$
R_j = median(r_{j1}, r_{j2}, r_{j3}, r_{j4})
$$

### Стабильность через IQR

$$
IQR_j = Q3(r_j)-Q1(r_j)
$$

Если IQR высокий, методы не согласны по признаку. Такой признак надо обсуждать отдельно или исключать из стабильного consensus-list.

## 8. Валидация без модели: MFF-NoFit

В старой версии MFF можно было считать через 1-NN-классификатор, но в строгом XDM-протоколе мы исключаем даже такой вариант.

Поэтому используем **MFF-NoFit**:

> консенсусные top-k признаки должны лучше сохранять структуру target в данных, чем случайные k признаков.

### Score разделимости

$$
S(F_k)=rac{mean\ distance\ between\ classes}{mean\ distance\ within\ classes}
$$

### MFF-NoFit

$$
MFF(k)=S(top	ext{-}k\ consensus)-\mathbb{E}[S(random\ k)]
$$

Интерпретация:

- `MFF > 0`: consensus-признаки лучше случайных сохраняют структуру target;
- `MFF ≈ 0`: consensus не лучше случайного выбора;
- `MFF < 0`: протокол ранжирования на этом датасете не подтверждён.

## 9. Что студент должен понимать перед защитой

1. XDM объясняет данные, а не обученную модель.
2. MI и Relief-style не доказывают причинность.
3. Геометрическая важность показывает разделимость, но может быть чувствительна к масштабированию.
4. Causal-screening — это не доказательство причинности, а проверка устойчивой условной зависимости.
5. Subgroup Discovery даёт интерпретируемые правила, но может переобучаться на случайные паттерны без permutation/bootstrap-проверки.
6. Консенсус нужен, потому что один метод почти всегда даёт неполную картину.
7. MFF-NoFit проверяет, что top-k признаки лучше случайных по структуре target, но не заменяет внешнюю экспертную валидацию.

## 10. Минимальная литература для обзора

Используйте эти направления в литературном обзоре статьи:

- Explainable Data Mining and interpretable knowledge discovery;
- Mutual Information and feature relevance;
- ReliefF and nearest-neighbor feature weighting;
- Conditional independence testing and PC-style causal discovery;
- Subgroup Discovery and exceptional model mining;
- Rank aggregation and consensus feature selection;
- Stability selection, bootstrap validation, permutation testing.

В статье важно подчеркнуть gap: большинство XAI-методов объясняют обученную модель, а MFEX/XDM предлагает протокол объяснения структуры табличных данных без обучаемого предиктора.